# Qwen2.5-3B-Instruct GSM8K LoRA SFT

This notebook trains `Qwen/Qwen2.5-3B-Instruct` with `LLaMA-Factory` on a Colab A100 using standard LoRA, and logs training to SwanLab.

Data source：
- `SFT_data_generation/SFT_data_generation/outputs/runs/20260402_201445/success.jsonl`

Pipeline summary:
- use 3000 teacher-generated GSM8K training examples
- fixed split with `seed=42`
- 10% validation split, i.e. 300 examples
- teacher system prompt is not included in student training inputs
- keep the checkpoint with the best `val_loss`, then run validation accuracy with the same answer extraction style used by the evaluation notebook


In [3]:
from google.colab import drive
drive.mount('/content/drive')
!rm -rf /content/project
!mkdir -p /content/project
!unzip -q /content/drive/MyDrive/project_sft.zip -d /content/project
!ls /content/project

Mounted at /content/drive
__MACOSX  SFT_data_generation  SFT_train


In [4]:
from pathlib import Path
import os
import sys
import json
import random
import re
import subprocess
from datetime import datetime

PROJECT_ROOT = Path('/content/project')
REPO_SFT_DIR = PROJECT_ROOT / 'SFT_data_generation'
SUCCESS_JSONL_PATH = REPO_SFT_DIR / 'outputs' / 'runs' / '20260426_132022' / 'success.jsonl'
WORKDIR = PROJECT_ROOT / 'SFT_train'
DATA_DIR = WORKDIR / 'data'
OUTPUT_DIR = WORKDIR / 'outputs' / 'qwen25_3b_gsm8k_lora_sft_full'
VAL_RESULTS_PATH = WORKDIR / 'val_results.json'
LLAMAFACTORY_DIR = Path('/content/LLaMA-Factory')

MODEL_ID = 'Qwen/Qwen2.5-3B-Instruct'
TRAIN_DATASET_NAME = 'gsm8k_sft_train'
EVAL_DATASET_NAME = 'gsm8k_sft_val'
SEED = 42
VAL_RATIO = 0.1
VAL_SIZE = 725
CUTOFF_LEN = 512
MAX_NEW_TOKENS = 512

NUM_TRAIN_EPOCHS = 3.0
LEARNING_RATE = 1e-5
LR_SCHEDULER_TYPE = 'cosine'
WARMUP_RATIO = 0.03
PER_DEVICE_TRAIN_BATCH_SIZE = 16
PER_DEVICE_EVAL_BATCH_SIZE = 16
GRADIENT_ACCUMULATION_STEPS = 1
WEIGHT_DECAY = 0.1
MAX_GRAD_NORM = 1.0
LORA_RANK = 32
LORA_ALPHA = 64
LORA_DROPOUT = 0.05
LOGGING_STEPS = 10
EVAL_STEPS = 25
SAVE_STEPS = 25
SAVE_TOTAL_LIMIT = 2
LORA_TARGET = 'q_proj,k_proj,v_proj,o_proj,up_proj,down_proj,gate_proj'

RUN_SMOKE_TEST = False
SMOKE_MAX_SAMPLES = 64
SMOKE_MAX_STEPS = 20

SWANLAB_SECRET_NAME = 'SWANLAB_API_KEY'
HF_TOKEN_SECRET_NAME = 'HF_TOKEN'
MANUAL_SWANLAB_API_KEY = ''
MANUAL_HF_TOKEN = ''
SWANLAB_PROJECT = 'gsm8k-lora-sft'
SWANLAB_RUN_NAME = 'qwen2.5-3b-gsm8k-lora-sft-full'
SWANLAB_WORKSPACE = ''

def read_colab_secret(secret_name: str) -> str:
    try:
        from google.colab import userdata
    except ImportError:
        return ''
    try:
        value = userdata.get(secret_name)
    except Exception:
        return ''
    return value or ''

SWANLAB_API_KEY = MANUAL_SWANLAB_API_KEY or read_colab_secret(SWANLAB_SECRET_NAME)
HF_TOKEN = MANUAL_HF_TOKEN or read_colab_secret(HF_TOKEN_SECRET_NAME)

ALL_DATA_PATH = DATA_DIR / 'gsm8k_sft_all.json'
TRAIN_DATA_PATH = DATA_DIR / 'gsm8k_sft_train.json'
VAL_DATA_PATH = DATA_DIR / 'gsm8k_sft_val.json'
DATASET_INFO_PATH = DATA_DIR / 'dataset_info.json'
TRAIN_CONFIG_PATH = WORKDIR / 'train_qwen25_3b_lora_sft.yaml'

print('PROJECT_ROOT      =', PROJECT_ROOT)
print('SUCCESS_JSONL     =', SUCCESS_JSONL_PATH)
print('WORKDIR           =', WORKDIR)
print('MODEL_ID          =', MODEL_ID)
print('RUN_SMOKE_TEST    =', RUN_SMOKE_TEST)
print('SWANLAB secret    =', bool(SWANLAB_API_KEY))
print('HF token secret   =', bool(HF_TOKEN))
assert SUCCESS_JSONL_PATH.exists(), f'Missing data file: {SUCCESS_JSONL_PATH}'


PROJECT_ROOT      = /content/project
SUCCESS_JSONL     = /content/project/SFT_data_generation/outputs/runs/20260426_132022/success.jsonl
WORKDIR           = /content/project/SFT_train
MODEL_ID          = Qwen/Qwen2.5-3B-Instruct
RUN_SMOKE_TEST    = False
SWANLAB secret    = False
HF token secret   = True


## Colab Setup

Recommended Drive layout:
```text
MyDrive/project/
  - qwen25_3b_llamafactory_lora_sft_colab.ipynb
  - SFT_data_generation/
    - SFT_data_generation/
      - outputs/
        - runs/
          - 20260426_132022/
            - success.jsonl
```

Then set:
- `PROJECT_ROOT` to the folder that contains `SFT_data_generation/` and `SFT_train/`
- create a Colab Secret named `SWANLAB_API_KEY`
- create a Colab Secret named `HF_TOKEN` if Hugging Face model download requires authenticatio


In [5]:
def run_cmd(cmd, cwd=None):
    print('>>>', ' '.join(cmd))
    subprocess.run(cmd, check=True, cwd=cwd)

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.parent.mkdir(parents=True, exist_ok=True)

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
    os.environ['HUGGINGFACEHUB_API_TOKEN'] = HF_TOKEN

if SWANLAB_API_KEY:
    os.environ['SWANLAB_API_KEY'] = SWANLAB_API_KEY

if not LLAMAFACTORY_DIR.exists():
    run_cmd(['git', 'clone', '--depth', '1', 'https://github.com/hiyouga/LLaMA-Factory.git', str(LLAMAFACTORY_DIR)])

run_cmd([sys.executable, '-m', 'pip', 'install', '-U', 'pip'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-e', f'{LLAMAFACTORY_DIR}[torch,metrics,swanlab]'])
run_cmd([sys.executable, '-m', 'pip', 'install', '-U', 'swanlab'])


>>> git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git /content/LLaMA-Factory
>>> /usr/bin/python3 -m pip install -U pip
>>> /usr/bin/python3 -m pip install -e /content/LLaMA-Factory[torch,metrics,swanlab]
>>> /usr/bin/python3 -m pip install -U swanlab


In [6]:
import yaml
from datasets import Dataset, load_dataset


def extract_index(example_id: str) -> int:
    return int(example_id.rsplit('_', 1)[1])


def build_student_output(reasoning: str, answer: str) -> str:
    reasoning = reasoning.strip()
    answer = answer.strip()
    return f"{reasoning}\n\nThe answer is {answer}."

records = []
with SUCCESS_JSONL_PATH.open('r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if not line:
            continue
        row = json.loads(line)
        records.append({
            'id': row['id'],
            'instruction': row['question'],
            'input': '',
            'output': build_student_output(row['response']['reasoning'], row['response']['answer']),
        })

records = sorted(records, key=lambda x: extract_index(x['id']))
all_dataset = Dataset.from_list(records)
split = all_dataset.train_test_split(test_size=VAL_SIZE, seed=SEED)
train_dataset = split['train']
val_dataset = split['test']

all_records = [all_dataset[i] for i in range(len(all_dataset))]
train_records = [train_dataset[i] for i in range(len(train_dataset))]
val_records = [val_dataset[i] for i in range(len(val_dataset))]

ALL_DATA_PATH.write_text(json.dumps(all_records, ensure_ascii=False, indent=2), encoding='utf-8')
TRAIN_DATA_PATH.write_text(json.dumps(train_records, ensure_ascii=False, indent=2), encoding='utf-8')
VAL_DATA_PATH.write_text(json.dumps(val_records, ensure_ascii=False, indent=2), encoding='utf-8')

# Save dataset_info for LLaMA-Factory.
dataset_info = {
    'gsm8k_sft_all': {
        'file_name': ALL_DATA_PATH.name,
        'columns': {
            'prompt': 'instruction',
            'query': 'input',
            'response': 'output',
        },
    },
    'gsm8k_sft_train': {
        'file_name': TRAIN_DATA_PATH.name,
        'columns': {
            'prompt': 'instruction',
            'query': 'input',
            'response': 'output',
        },
    },
    'gsm8k_sft_val': {
        'file_name': VAL_DATA_PATH.name,
        'columns': {
            'prompt': 'instruction',
            'query': 'input',
            'response': 'output',
        },
    },
}
DATASET_INFO_PATH.write_text(json.dumps(dataset_info, ensure_ascii=False, indent=2), encoding='utf-8')

print('all  =', len(all_records))
print('train=', len(train_records))
print('val  =', len(val_records))
print('train file:', TRAIN_DATA_PATH)
print('val file  :', VAL_DATA_PATH)
print('dataset_info:', DATASET_INFO_PATH)

for sample in train_records[:3]:
    print('-' * 80)
    print(sample['id'])
    print(sample['instruction'][:200])
    print(sample['output'][-200:])


all  = 7256
train= 6531
val  = 725
train file: /content/project/SFT_train/data/gsm8k_sft_train.json
val file  : /content/project/SFT_train/data/gsm8k_sft_val.json
dataset_info: /content/project/SFT_train/data/dataset_info.json
--------------------------------------------------------------------------------
gsm8k_train_02980
At his craftwork store, Howard has a collection of 70 wooden bowls where he rewards two to his customers for every 10 they buy. If he had 20 customers that day, half of whom bought 20 bowls each, calc
 half (10) bought 20 bowls each. For every 10 bought, 2 rewards: 20 bowls = 2 sets of 10, so 2*2=4 rewards per customer. Total rewards: 10 customers *4=40. Remaining bowls:70-40=30.

The answer is 30.
--------------------------------------------------------------------------------
gsm8k_train_05777
For the family reunion, Peter is buying 16 pounds of bone-in chicken and half that amount in hamburgers.  He's going to buy 2 more pounds of hot dogs than hamburgers.  He's 

In [7]:
from statistics import mean
from transformers import AutoTokenizer


def pct(values, ratio):
    values = sorted(values)
    return values[int((len(values) - 1) * ratio)]


tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
lengths = []

for item in all_records:
    messages = [
        {"role": "user", "content": item["instruction"]},
        {"role": "assistant", "content": item["output"]},
    ]
    tokenized = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=False,
        return_dict=True,
        return_tensors="pt",
    )
    lengths.append(int(tokenized["input_ids"].shape[-1]))

stats = {
    "mean": round(mean(lengths), 2),
    "p50": pct(lengths, 0.50),
    "p90": pct(lengths, 0.90),
    "p95": pct(lengths, 0.95),
    "p99": pct(lengths, 0.99),
    "max": max(lengths),
}
print(stats)
assert stats["max"] <= CUTOFF_LEN, f"Max token length {stats['max']} exceeds cutoff_len {CUTOFF_LEN}"


config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'mean': 180.84, 'p50': 175, 'p90': 235, 'p95': 254, 'p99': 293, 'max': 459}


In [8]:
base_config = {
    'model_name_or_path': MODEL_ID,
    'trust_remote_code': True,
    'stage': 'sft',
    'do_train': True,
    'do_eval': True,
    'finetuning_type': 'lora',
    'lora_target': LORA_TARGET,
    'lora_rank': LORA_RANK,
    'lora_alpha': LORA_ALPHA,
    'lora_dropout': LORA_DROPOUT,
    'template': 'qwen',
    'dataset_dir': str(DATA_DIR),
    'dataset': TRAIN_DATASET_NAME,
    'eval_dataset': EVAL_DATASET_NAME,
    'cutoff_len': CUTOFF_LEN,
    'overwrite_cache': True,
    'preprocessing_num_workers': 2,
    'output_dir': str(OUTPUT_DIR),
    'logging_steps': LOGGING_STEPS,
    'save_steps': SAVE_STEPS,
    'save_strategy': 'steps',
    'save_total_limit': SAVE_TOTAL_LIMIT,
    'plot_loss': True,
    'overwrite_output_dir': True,
    'per_device_train_batch_size': PER_DEVICE_TRAIN_BATCH_SIZE,
    'per_device_eval_batch_size': PER_DEVICE_EVAL_BATCH_SIZE,
    'gradient_accumulation_steps': GRADIENT_ACCUMULATION_STEPS,
    'learning_rate': LEARNING_RATE,
    'num_train_epochs': NUM_TRAIN_EPOCHS,
    'lr_scheduler_type': LR_SCHEDULER_TYPE,
    'warmup_ratio': WARMUP_RATIO,
    'weight_decay': WEIGHT_DECAY,
    'max_grad_norm': MAX_GRAD_NORM,
    'bf16': True,
    'gradient_checkpointing': True,
    'gradient_checkpointing_kwargs': {'use_reentrant': False},
    'ddp_timeout': 180000000,
    'optim': 'adamw_torch',
    'packing': False,
    'flash_attn': 'auto',
    'seed': SEED,
    'eval_strategy': 'steps',
    'eval_steps': EVAL_STEPS,
    'load_best_model_at_end': True,
    'metric_for_best_model': 'eval_loss',
    'greater_is_better': False,
    'save_only_model': True,
    'use_swanlab': bool(SWANLAB_API_KEY),
    'swanlab_project': SWANLAB_PROJECT,
    'swanlab_run_name': SWANLAB_RUN_NAME,
}

if SWANLAB_WORKSPACE:
    base_config['swanlab_workspace'] = SWANLAB_WORKSPACE

if RUN_SMOKE_TEST:
    base_config['max_samples'] = SMOKE_MAX_SAMPLES
    base_config['max_steps'] = SMOKE_MAX_STEPS

config_to_dump = {k: v for k, v in base_config.items() if v is not None and v != ''}
TRAIN_CONFIG_PATH.write_text(yaml.safe_dump(config_to_dump, allow_unicode=True, sort_keys=False), encoding='utf-8')
print(TRAIN_CONFIG_PATH.read_text(encoding='utf-8'))


model_name_or_path: Qwen/Qwen2.5-3B-Instruct
trust_remote_code: true
stage: sft
do_train: true
do_eval: true
finetuning_type: lora
lora_target: q_proj,k_proj,v_proj,o_proj,up_proj,down_proj,gate_proj
lora_rank: 32
lora_alpha: 64
lora_dropout: 0.05
template: qwen
dataset_dir: /content/project/SFT_train/data
dataset: gsm8k_sft_train
eval_dataset: gsm8k_sft_val
cutoff_len: 512
overwrite_cache: true
preprocessing_num_workers: 2
output_dir: /content/project/SFT_train/outputs/qwen25_3b_gsm8k_lora_sft_full
logging_steps: 10
save_steps: 25
save_strategy: steps
save_total_limit: 2
plot_loss: true
overwrite_output_dir: true
per_device_train_batch_size: 16
per_device_eval_batch_size: 16
gradient_accumulation_steps: 1
learning_rate: 1.0e-05
num_train_epochs: 3.0
lr_scheduler_type: cosine
warmup_ratio: 0.03
weight_decay: 0.1
max_grad_norm: 1.0
bf16: true
gradient_checkpointing: true
gradient_checkpointing_kwargs:
  use_reentrant: false
ddp_timeout: 180000000
optim: adamw_torch
packing: false
flash_

In [9]:
train_cmd = ['llamafactory-cli', 'train', str(TRAIN_CONFIG_PATH)]
print('Running training...')
run_cmd(train_cmd, cwd=LLAMAFACTORY_DIR)


Running training...
>>> llamafactory-cli train /content/project/SFT_train/train_qwen25_3b_lora_sft.yaml


In [10]:
trainer_state_path = OUTPUT_DIR / 'trainer_state.json'
assert trainer_state_path.exists(), f'Missing trainer state: {trainer_state_path}'
trainer_state = json.loads(trainer_state_path.read_text(encoding='utf-8'))
best_checkpoint = Path(trainer_state['best_model_checkpoint'])
best_val_loss = trainer_state['best_metric']
assert best_checkpoint.exists(), f'Missing best checkpoint: {best_checkpoint}'
print('best_checkpoint =', best_checkpoint)
print('best_val_loss   =', best_val_loss)


best_checkpoint = /content/project/SFT_train/outputs/qwen25_3b_gsm8k_lora_sft_full/checkpoint-1200
best_val_loss   = 0.28411665558815


In [11]:
SYSTEM_PROMPT_EVAL = (
    ''
)
import time
import traceback
import torch
from tqdm.auto import tqdm
from peft import PeftModel
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset

EVAL_GENERATION_BATCH_SIZE = 64


def extract_answer(text: str):
    match = re.search(r"####\s*([\d,.\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    match = re.search(r"[Tt]he answer is[:\s]*([\d,.\-]+)", text)
    if match:
        return match.group(1).replace(",", "").strip()

    numbers = re.findall(r"[\d,]+\.?\d*", text)
    if numbers:
        return numbers[-1].replace(",", "").strip()

    return None


def is_correct(predicted, gold):
    if predicted is None:
        return False
    try:
        return abs(float(predicted) - float(gold)) < 1e-6
    except ValueError:
        return predicted.strip().lower() == gold.strip().lower()


def build_messages(question: str):
    return [
        {"role": "system", "content": SYSTEM_PROMPT_EVAL},
        {"role": "user", "content": question},
    ]


def official_gold(answer_text: str) -> str:
    return extract_answer(answer_text)


print(f"Loading tokenizer: {MODEL_ID}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

dtype = torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported() else torch.float16
print(f"Loading base model: {MODEL_ID}")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)

print(f"Loading adapter: {best_checkpoint}")
model = PeftModel.from_pretrained(model_base, str(best_checkpoint))
model.eval()
DEVICE = next(model.parameters()).device
print("Model loaded on:", DEVICE)

val_records = json.loads(VAL_DATA_PATH.read_text(encoding="utf-8"))
gsm8k_train = load_dataset("openai/gsm8k", "main", split="train")

results = []
correct = 0
errors = 0
start_all = time.perf_counter()

for start_idx in tqdm(range(0, len(val_records), EVAL_GENERATION_BATCH_SIZE), desc="Validating best checkpoint"):
    batch = val_records[start_idx:start_idx + EVAL_GENERATION_BATCH_SIZE]
    questions = [item["instruction"] for item in batch]
    ids = [item["id"] for item in batch]
    golds = [official_gold(gsm8k_train[int(item["id"].rsplit("_", 1)[1])]["answer"]) for item in batch]
    messages_batch = [build_messages(q) for q in questions]

    try:
        tokenized = tokenizer.apply_chat_template(
            messages_batch,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            padding=True,
        )
        input_ids = tokenized["input_ids"].to(DEVICE)
        attention_mask = tokenized["attention_mask"].to(DEVICE)
        prompt_lengths = attention_mask.sum(dim=1).tolist()

        t0 = time.perf_counter()
        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        latency = time.perf_counter() - t0

        for i, item in enumerate(batch):
            new_tokens = output_ids[i][int(prompt_lengths[i]):]
            response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            predicted = extract_answer(response)
            ok = is_correct(predicted, golds[i])

            if ok:
                correct += 1

            results.append({
                "id": ids[i],
                "question": questions[i],
                "gold": golds[i],
                "predicted": predicted,
                "correct": ok,
                "response": response,
                "prompt_tokens": int(prompt_lengths[i]),
                "latency_s": round(latency / len(batch), 3),
            })

    except Exception:
        error_text = traceback.format_exc()
        errors += len(batch)

        for i, item in enumerate(batch):
            results.append({
                "id": ids[i],
                "question": questions[i],
                "gold": golds[i],
                "predicted": None,
                "correct": False,
                "response": error_text,
                "prompt_tokens": 0,
                "latency_s": 0.0,
            })

summary = {
    "model": MODEL_ID,
    "best_checkpoint": str(best_checkpoint),
    "best_val_loss": float(best_val_loss),
    "total": len(val_records),
    "correct": correct,
    "errors": errors,
    "accuracy": round(correct / len(val_records) * 100, 2),
    "total_latency_s": round(time.perf_counter() - start_all, 2),
    "eval_batch_size": EVAL_GENERATION_BATCH_SIZE,
}

VAL_RESULTS_PATH.write_text(
    json.dumps({"summary": summary, "results": results}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(summary, ensure_ascii=False, indent=2))
print("val results saved to:", VAL_RESULTS_PATH)
print("\nFirst 5 wrong examples:")
for row in [x for x in results if not x["correct"]][:5]:
    print("-" * 80)
    print("id       :", row["id"])
    print("gold     :", row["gold"])
    print("predicted:", row["predicted"])
    print("response :", row["response"][:5000])


Loading tokenizer: Qwen/Qwen2.5-3B-Instruct
Loading base model: Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Loading adapter: /content/project/SFT_train/outputs/qwen25_3b_gsm8k_lora_sft_full/checkpoint-1200
Model loaded on: cuda:0


Validating best checkpoint:   0%|          | 0/12 [00:00<?, ?it/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


{
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "best_checkpoint": "/content/project/SFT_train/outputs/qwen25_3b_gsm8k_lora_sft_full/checkpoint-1200",
  "best_val_loss": 0.28411665558815,
  "total": 725,
  "correct": 603,
  "errors": 0,
  "accuracy": 83.17,
  "total_latency_s": 310.51,
  "eval_batch_size": 64
}
val results saved to: /content/project/SFT_train/val_results.json

First 5 wrong examples:
--------------------------------------------------------------------------------
id       : gsm8k_train_03178
gold     : 25550
predicted: 70.
response : - Calculate the total circuits run per day: 7 + 3 = 10 circuits.
   - Multiply the daily total by the number of days in a week (7): 10 * 7 = 70.

The answer is 70.
--------------------------------------------------------------------------------
id       : gsm8k_train_03601
gold     : 112
predicted: 104.
response : First, calculate the water needed for the first two tanks: 2 tanks * 8 gallons = 16 gallons. Next, determine the water needed for th

In [12]:
BASELINE_VAL_RESULTS_PATH = WORKDIR / 'base_model_val_results.json'

# Release the LoRA model before loading the base model again.
del model
try:
    del model_base
except NameError:
    pass
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Loading original base model for baseline validation: {MODEL_ID}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()
BASE_DEVICE = next(base_model.parameters()).device
print("Base model loaded on:", BASE_DEVICE)

base_results = []
base_correct = 0
base_errors = 0
base_start_all = time.perf_counter()

for start_idx in tqdm(range(0, len(val_records), EVAL_GENERATION_BATCH_SIZE), desc="Validating original base model"):
    batch = val_records[start_idx:start_idx + EVAL_GENERATION_BATCH_SIZE]
    questions = [item["instruction"] for item in batch]
    ids = [item["id"] for item in batch]
    golds = [official_gold(gsm8k_train[int(item["id"].rsplit("_", 1)[1])]["answer"]) for item in batch]
    messages_batch = [build_messages(q) for q in questions]

    try:
        tokenized = tokenizer.apply_chat_template(
            messages_batch,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            padding=True,
        )
        input_ids = tokenized["input_ids"].to(BASE_DEVICE)
        attention_mask = tokenized["attention_mask"].to(BASE_DEVICE)
        prompt_lengths = attention_mask.sum(dim=1).tolist()

        t0 = time.perf_counter()
        with torch.no_grad():
            output_ids = base_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        latency = time.perf_counter() - t0

        for i, item in enumerate(batch):
            new_tokens = output_ids[i][int(prompt_lengths[i]):]
            response = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            predicted = extract_answer(response)
            ok = is_correct(predicted, golds[i])

            if ok:
                base_correct += 1

            base_results.append({
                "id": ids[i],
                "question": questions[i],
                "gold": golds[i],
                "predicted": predicted,
                "correct": ok,
                "response": response,
                "prompt_tokens": int(prompt_lengths[i]),
                "latency_s": round(latency / len(batch), 3),
            })

    except Exception:
        error_text = traceback.format_exc()
        base_errors += len(batch)

        for i, item in enumerate(batch):
            base_results.append({
                "id": ids[i],
                "question": questions[i],
                "gold": golds[i],
                "predicted": None,
                "correct": False,
                "response": error_text,
                "prompt_tokens": 0,
                "latency_s": 0.0,
            })

base_summary = {
    "model": MODEL_ID,
    "checkpoint": "base_model",
    "total": len(val_records),
    "correct": base_correct,
    "errors": base_errors,
    "accuracy": round(base_correct / len(val_records) * 100, 2),
    "total_latency_s": round(time.perf_counter() - base_start_all, 2),
    "eval_batch_size": EVAL_GENERATION_BATCH_SIZE,
}

BASELINE_VAL_RESULTS_PATH.write_text(
    json.dumps({"summary": base_summary, "results": base_results}, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(base_summary, ensure_ascii=False, indent=2))
print("base val results saved to:", BASELINE_VAL_RESULTS_PATH)

print("First 5 wrong examples from original base model:")
for row in [x for x in base_results if not x["correct"]][:5]:
    print("-" * 80)
    print("id       :", row["id"])
    print("gold     :", row["gold"])
    print("predicted:", row["predicted"])
    print("response :", row["response"][:5000])


Loading original base model for baseline validation: Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Base model loaded on: cuda:0


Validating original base model:   0%|          | 0/12 [00:00<?, ?it/s]

{
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "checkpoint": "base_model",
  "total": 725,
  "correct": 636,
  "errors": 0,
  "accuracy": 87.72,
  "total_latency_s": 290.58,
  "eval_batch_size": 64
}
base val results saved to: /content/project/SFT_train/base_model_val_results.json
First 5 wrong examples from original base model:
--------------------------------------------------------------------------------
id       : gsm8k_train_00024
gold     : 62
predicted: None
response : 
--------------------------------------------------------------------------------
id       : gsm8k_train_04152
gold     : 120
predicted: 30
response : cost of the more expensive coat per year is calculated as follows: $300 / 15 years = $20 per year.

The less expensive coat costs $120 and lasts for 5 years, so its annual cost is $120 / 5 years = $24 per year.

If Karen chooses the more expensive coat, she pays $20 per year.
If Karen chooses the cheaper coat, she pays $24 per year.

The difference in cost per year bet

In [13]:
if SWANLAB_API_KEY:
    import swanlab

    swanlab.login(api_key=SWANLAB_API_KEY)
    eval_run = swanlab.init(
        project=SWANLAB_PROJECT,
        experiment_name=f'{SWANLAB_RUN_NAME}-validation',
        workspace=SWANLAB_WORKSPACE or None,
        config={
            'model_name_or_path': MODEL_ID,
            'best_checkpoint': str(best_checkpoint),
            'best_val_loss': float(best_val_loss),
            'validation_size': len(val_records),
        },
    )
    eval_run.log({'val_accuracy': float(summary['accuracy'])})
    eval_run.finish()
    print('Logged validation accuracy to SwanLab.')
else:
    print('SWANLAB_API_KEY is empty, skipping manual SwanLab eval logging.')


SWANLAB_API_KEY is empty, skipping manual SwanLab eval logging.


## Notes

- training loss and evaluation loss are logged by LLaMA-Factory and SwanLab during training
- after training, the notebook creates a separate SwanLab validation run and logs `val_accuracy` and `best_checkpoint`
- for a quick smoke test, set `RUN_SMOKE_TEST = True`; this limits training to a small subset and a small number of steps
- validation generation and answer extraction are aligned with `gsm8k_student_distill.ipynb`


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
!mkdir -p "/content/drive/MyDrive/hpml_instruct-gsm8k-lora-sft"
!cp -r /content/project "/content/drive/MyDrive/hpml_instruct-gsm8k-lora-sft/"


## Official GSM8K Test Set Evaluation

Evaluates both the **SFT distilled model** and the **baseline model** on the official GSM8K test split (1319 examples) to produce the final quantitative results for the report.

### Two evaluation cells below:
- **SFT model**: Reloads the best LoRA checkpoint (`checkpoint-1200`) and runs greedy inference on all 1319 test examples
- **Baseline model**: Releases the SFT model, reloads the original `Qwen2.5-3B-Instruct` without any adapter, and runs the same evaluation

In [20]:
import torch
import traceback
import time
from tqdm.auto import tqdm
from peft import PeftModel
from transformers import AutoModelForCausalLM
from datasets import load_dataset

# Reload SFT model
dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    else torch.float16
)

try:
    del base_model
    torch.cuda.empty_cache()
except NameError:
    pass

print(f"Reloading base model: {MODEL_ID}")
model_base = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
print(f"Reloading LoRA adapter: {best_checkpoint}")
model = PeftModel.from_pretrained(model_base, str(best_checkpoint))
model.eval()
DEVICE = next(model.parameters()).device
print("SFT model reloaded on:", DEVICE)

# Load official test set
gsm8k_test = load_dataset("openai/gsm8k", "main", split="test")
print(f"Test set size: {len(gsm8k_test)} examples")

# Evaluation loop
results_test = []
correct_test = 0
errors_test  = 0
start_all    = time.perf_counter()

for start_idx in tqdm(
    range(0, len(gsm8k_test), EVAL_GENERATION_BATCH_SIZE),
    desc="GSM8K test set",
):
    batch      = gsm8k_test[start_idx : start_idx + EVAL_GENERATION_BATCH_SIZE]
    questions  = batch["question"]
    golds      = [official_gold(a) for a in batch["answer"]]
    msgs_batch = [build_messages(q) for q in questions]

    try:
        tokenized = tokenizer.apply_chat_template(
            msgs_batch,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            padding=True,
        )
        input_ids      = tokenized["input_ids"].to(DEVICE)
        attention_mask = tokenized["attention_mask"].to(DEVICE)
        prompt_lengths = attention_mask.sum(dim=1).tolist()

        t0 = time.perf_counter()
        with torch.no_grad():
            output_ids = model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        latency = time.perf_counter() - t0

        for i in range(len(questions)):
            new_tokens = output_ids[i][int(prompt_lengths[i]):]
            response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            predicted  = extract_answer(response)
            ok         = is_correct(predicted, golds[i])
            if ok:
                correct_test += 1
            results_test.append({
                "question":      questions[i],
                "gold":          golds[i],
                "predicted":     predicted,
                "correct":       ok,
                "response":      response,
                "prompt_tokens": int(prompt_lengths[i]),
                "latency_s":     round(latency / len(questions), 3),
            })

    except Exception:
        error_text = traceback.format_exc()
        errors_test += len(questions)
        for i in range(len(questions)):
            results_test.append({
                "question":      questions[i],
                "gold":          golds[i],
                "predicted":     None,
                "correct":       False,
                "response":      error_text,
                "prompt_tokens": 0,
                "latency_s":     0.0,
            })

# Summary
test_summary = {
    "model":           MODEL_ID,
    "checkpoint":      str(best_checkpoint),
    "split":           "test",
    "total":           len(gsm8k_test),
    "correct":         correct_test,
    "errors":          errors_test,
    "accuracy":        round(correct_test / len(gsm8k_test) * 100, 2),
    "total_latency_s": round(time.perf_counter() - start_all, 2),
    "eval_batch_size": EVAL_GENERATION_BATCH_SIZE,
}

TEST_RESULTS_PATH = WORKDIR / "test_results_sft.json"
TEST_RESULTS_PATH.write_text(
    json.dumps({"summary": test_summary, "results": results_test},
               ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(test_summary, indent=2))
print("\nSaved to", TEST_RESULTS_PATH)

print("\nFirst 5 wrong examples:")
for row in [x for x in results_test if not x["correct"]][:5]:
    print("-" * 80)
    print("gold     :", row["gold"])
    print("predicted:", row["predicted"])
    print("response :", row["response"][:300])

Reloading base model: Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Reloading LoRA adapter: /content/project/SFT_train/outputs/qwen25_3b_gsm8k_lora_sft_full/checkpoint-1200
SFT model reloaded on: cuda:0
Test set size: 1319 examples


GSM8K test set:   0%|          | 0/21 [00:00<?, ?it/s]

{
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "checkpoint": "/content/project/SFT_train/outputs/qwen25_3b_gsm8k_lora_sft_full/checkpoint-1200",
  "split": "test",
  "total": 1319,
  "correct": 1012,
  "errors": 0,
  "accuracy": 76.72,
  "total_latency_s": 566.15,
  "eval_batch_size": 64
}

Saved → /content/project/SFT_train/test_results_sft.json

First 5 wrong examples:
--------------------------------------------------------------------------------
gold     : 20
predicted: 140.
response : The total daily feed requirement for one chicken is 3 cups per meal multiplied by 3 meals, which equals 9 cups. For a flock of 20 chickens, the total daily feed required is 9 cups multiplied by 20 chickens, equaling 180 cups. The sum of the morning and afternoon feed amounts is 15 cups plus 25 cups,
--------------------------------------------------------------------------------
gold     : 160
predicted: 110.
response : - The total file size is 200 GB.
- The normal download speed is 2 GB/minute.
- The r

In [21]:
# Baseline evaluation on official test set
# Free SFT model
del model
del model_base
torch.cuda.empty_cache()

print(f"Loading base model (no adapter): {MODEL_ID}")
base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=dtype,
    device_map="auto",
    trust_remote_code=True,
)
base_model.eval()
BASE_DEVICE = next(base_model.parameters()).device
print("Base model loaded on:", BASE_DEVICE)

results_base_test = []
correct_base_test = 0
errors_base_test  = 0
start_all = time.perf_counter()

for start_idx in tqdm(
    range(0, len(gsm8k_test), EVAL_GENERATION_BATCH_SIZE),
    desc="GSM8K test set (baseline)",
):
    batch      = gsm8k_test[start_idx : start_idx + EVAL_GENERATION_BATCH_SIZE]
    questions  = batch["question"]
    golds      = [official_gold(a) for a in batch["answer"]]
    msgs_batch = [build_messages(q) for q in questions]

    try:
        tokenized = tokenizer.apply_chat_template(
            msgs_batch,
            tokenize=True,
            add_generation_prompt=True,
            return_tensors="pt",
            return_dict=True,
            padding=True,
        )
        input_ids      = tokenized["input_ids"].to(BASE_DEVICE)
        attention_mask = tokenized["attention_mask"].to(BASE_DEVICE)
        prompt_lengths = attention_mask.sum(dim=1).tolist()

        t0 = time.perf_counter()
        with torch.no_grad():
            output_ids = base_model.generate(
                input_ids=input_ids,
                attention_mask=attention_mask,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        latency = time.perf_counter() - t0

        for i in range(len(questions)):
            new_tokens = output_ids[i][int(prompt_lengths[i]):]
            response   = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
            predicted  = extract_answer(response)
            ok         = is_correct(predicted, golds[i])
            if ok:
                correct_base_test += 1
            results_base_test.append({
                "question":      questions[i],
                "gold":          golds[i],
                "predicted":     predicted,
                "correct":       ok,
                "response":      response,
                "prompt_tokens": int(prompt_lengths[i]),
                "latency_s":     round(latency / len(questions), 3),
            })

    except Exception:
        error_text = traceback.format_exc()
        errors_base_test += len(questions)
        for i in range(len(questions)):
            results_base_test.append({
                "question":  questions[i],
                "gold":      golds[i],
                "predicted": None,
                "correct":   False,
                "response":  error_text,
                "prompt_tokens": 0,
                "latency_s": 0.0,
            })

base_test_summary = {
    "model":           MODEL_ID,
    "checkpoint":      "base_model",
    "split":           "test",
    "total":           len(gsm8k_test),
    "correct":         correct_base_test,
    "errors":          errors_base_test,
    "accuracy":        round(correct_base_test / len(gsm8k_test) * 100, 2),
    "total_latency_s": round(time.perf_counter() - start_all, 2),
    "eval_batch_size": EVAL_GENERATION_BATCH_SIZE,
}

BASE_TEST_RESULTS_PATH = WORKDIR / "test_results_baseline.json"
BASE_TEST_RESULTS_PATH.write_text(
    json.dumps({"summary": base_test_summary, "results": results_base_test},
               ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(json.dumps(base_test_summary, indent=2))
print("\nSaved to", BASE_TEST_RESULTS_PATH)

# Final comparison
print("\n" + "=" * 50)
print(f"  GSM8K Official Test Set ({len(gsm8k_test)} examples)")
print("=" * 50)
print(f"  Baseline : {correct_base_test:>4d} / {len(gsm8k_test)} = {base_test_summary['accuracy']:>6.2f}%")
print(f"  SFT      : {1012:>4d} / {len(gsm8k_test)} = {'76.72':>6}%")
print(f"  Delta    : {round(76.72 - base_test_summary['accuracy'], 2):>+.2f} pp")
print("=" * 50)

Loading base model (no adapter): Qwen/Qwen2.5-3B-Instruct


Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

Base model loaded on: cuda:0


GSM8K test set (baseline):   0%|          | 0/21 [00:00<?, ?it/s]

{
  "model": "Qwen/Qwen2.5-3B-Instruct",
  "checkpoint": "base_model",
  "split": "test",
  "total": 1319,
  "correct": 1016,
  "errors": 0,
  "accuracy": 77.03,
  "total_latency_s": 522.74,
  "eval_batch_size": 64
}

Saved → /content/project/SFT_train/test_results_baseline.json

  GSM8K Official Test Set (1319 examples)
  Baseline : 1016 / 1319 =  77.03%
  SFT      : 1012 / 1319 =  76.72%
  Delta    : -0.31 pp
